In [1]:
TRAIN_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
TEST_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
MODEL_PATH = "/kaggle/input/aes2-debertav3large/transformers/default/1"
OUTPUT_PATH = "/kaggle/working/"

MAX_LEN = 512
OVERLAP = 128
MIN_CHUNK_RATIO = 0.375  # Skip if this chunk < RATIO * MAX_LEN
BATCH_SIZE = 2
NUM_LABELS = 6



In [2]:
import os

os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.metrics import cohen_kappa_score, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())




Torch: 2.6.0+cu124
CUDA available: True


In [3]:
def _resolve_local_model_dir(path: str) -> str:
    """
    Resolve a directory that contains config.json (or return original path if not found).
    """
    path = os.path.expanduser(path)
    if os.path.isfile(path):
        return os.path.dirname(path)
    if os.path.isdir(path) and os.path.isfile(os.path.join(path, "config.json")):
        return path

    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            if "config.json" in files:
                return root

    return path


def _is_transformers_checkpoint_dir(d: str) -> bool:
    if not os.path.isdir(d):
        return False
    if not os.path.isfile(os.path.join(d, "config.json")):
        return False
    weight_files = [
        "pytorch_model.bin",
        "model.safetensors",
        "pytorch_model.bin.index.json",
        "model.safetensors.index.json",
    ]
    return any(os.path.isfile(os.path.join(d, wf)) for wf in weight_files)


def _safe_read_num_labels(d: str):
    """
    Prefer checkpoints that actually have a 6-label head to avoid loading unrelated cached models.
    """
    try:
        cfg = AutoConfig.from_pretrained(d, local_files_only=True)
        return getattr(cfg, "num_labels", None)
    except Exception:
        return None


def _find_any_checkpoint_under(base_dir: str, max_hits: int = 200):
    """
    Robustly search for local HF checkpoint dirs that have config.json + weights.
    """
    hits = []
    if not os.path.isdir(base_dir):
        return hits
    for root, dirs, files in os.walk(base_dir):
        if "config.json" in files and _is_transformers_checkpoint_dir(root):
            hits.append(root)
            if len(hits) >= max_hits:
                break
    return hits


def _score_checkpoint_path(p: str, desired_num_labels: int = 6) -> int:
    """
    Rank checkpoints to more likely pick an AES finetune.
    """
    pl = p.lower()
    score = 0

    nl = _safe_read_num_labels(p)
    if nl == desired_num_labels:
        score += 300
    elif nl is None:
        score += 30
    else:
        score -= 50

    for key, w in [
        ("learning-agency", 40),
        ("automated-essay-scoring", 30),
        ("essay", 20),
        ("aes2", 20),
        ("aes", 10),
        ("deberta", 10),
        ("debertav3", 12),
        ("transformers/default", 20),
        ("/transformers/default/1", 40),
        (os.path.sep + "transformers" + os.path.sep, 8),
        (os.path.sep + "default" + os.path.sep, 8),
        (os.path.sep + "1", 3),
        ("checkpoint", 2),
    ]:
        if key in pl:
            score += w

    for bad, w in [
        ("tokenizer", -5),
        ("vocab", -5),
        ("onnx", -8),
        ("logs", -5),
        ("wandb", -5),
    ]:
        if bad in pl:
            score += w

    return score


def _pick_model_dir(primary_path: str) -> str | None:
    primary_path_resolved = _resolve_local_model_dir(primary_path)
    if _is_transformers_checkpoint_dir(primary_path_resolved):
        return primary_path_resolved

    preferred_roots = [
        "/kaggle/input/learning-agency-lab-automated-essay-scoring-2",
        "/kaggle/input",
    ]

    candidates = [
        primary_path,
        primary_path_resolved,
        "/kaggle/input/aes2-debertav3large/transformers/default/1",
    ]
    for cand in candidates:
        resolved = _resolve_local_model_dir(cand)
        if _is_transformers_checkpoint_dir(resolved):
            return resolved

    hits = []
    for base in preferred_roots:
        hits.extend(_find_any_checkpoint_under(base_dir=base, max_hits=1200))
        if len(hits) >= 1200:
            break
    hits = list(dict.fromkeys(hits))
    if not hits:
        return None

    hits_sorted = sorted(
        hits,
        key=lambda p: (_score_checkpoint_path(p, desired_num_labels=NUM_LABELS), p),
        reverse=True,
    )
    return hits_sorted[0]


MODEL_DIR = _pick_model_dir(MODEL_PATH)
print("Resolved MODEL_DIR (auto-discovered):", MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)



Resolved MODEL_DIR (auto-discovered): None
Device: cuda


In [4]:
FALLBACK_MODEL_CANDIDATES = [
    "microsoft/deberta-v3-base",
    "microsoft/deberta-v3-large",
    "microsoft/deberta-base",
    "microsoft/deberta-large",
]

tokenizer = None
model = None
MODEL_AVAILABLE = False
MODEL_SOURCE = None
FORCE_HEAD_FIT = False  # will be set True if we loaded a non-6-label head
MODEL_LOAD_REF = (
    None  # str path or HF name (offline-cached); used for head-fit construction
)

load_errors = []

# Change (score-improving): always force a 6-label config at load time if possible.
# This prevents falling into the weak scalar-calibration branch and enables argmax 1..6 predictions.
if MODEL_DIR is not None:
    try:
        cfg = AutoConfig.from_pretrained(
            MODEL_DIR, local_files_only=True, num_labels=NUM_LABELS
        )

        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_DIR, local_files_only=True, use_fast=True
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_DIR,
            config=cfg,
            local_files_only=True,
            ignore_mismatched_sizes=True,
        )

        MODEL_AVAILABLE = True
        MODEL_SOURCE = f"local_dir:{MODEL_DIR}"
        MODEL_LOAD_REF = MODEL_DIR
        print("Loaded model from local directory:", MODEL_DIR)
        print(
            "Checkpoint config num_labels (as loaded):",
            getattr(model.config, "num_labels", None),
        )

        if getattr(model.config, "num_labels", None) != NUM_LABELS:
            FORCE_HEAD_FIT = True
            print(
                "Loaded checkpoint is not 6-label; will trigger 6-label head fitting fallback."
            )
        else:
            # Keep your existing behavior: still allow head-fitting to make it task-relevant if needed.
            FORCE_HEAD_FIT = False
    except Exception as e:
        load_errors.append((f"local_dir:{MODEL_DIR}", repr(e)))
        MODEL_AVAILABLE = False
        MODEL_SOURCE = None
        FORCE_HEAD_FIT = False
        MODEL_LOAD_REF = None

if not MODEL_AVAILABLE:
    for name in FALLBACK_MODEL_CANDIDATES:
        try:
            cfg = AutoConfig.from_pretrained(
                name, local_files_only=True, num_labels=NUM_LABELS
            )

            tokenizer = AutoTokenizer.from_pretrained(
                name, local_files_only=True, use_fast=True
            )
            model = AutoModelForSequenceClassification.from_pretrained(
                name,
                config=cfg,
                local_files_only=True,
                ignore_mismatched_sizes=True,
            )
            MODEL_AVAILABLE = True
            MODEL_SOURCE = f"fallback_name:{name}"
            MODEL_LOAD_REF = name
            print("Loaded fallback pretrained model (offline cache):", name)
            print(
                "Fallback num_labels (as loaded):",
                getattr(model.config, "num_labels", None),
            )
            # Keep your existing approach: fit to make predictions task-aligned rather than constant.
            FORCE_HEAD_FIT = True
            break
        except Exception as e:
            load_errors.append((f"fallback_name:{name}", repr(e)))
            MODEL_AVAILABLE = False
            MODEL_SOURCE = None
            FORCE_HEAD_FIT = False
            MODEL_LOAD_REF = None

if not MODEL_AVAILABLE:
    print("Model/tokenizer loading failed. Errors tried (most recent last):")
    for tag, err in load_errors[-10:]:
        print(f" - {tag}: {err}")

if MODEL_AVAILABLE:
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))

    model = model.to(device)
    model.eval()

    print("Loaded tokenizer:", type(tokenizer).__name__)
    print("Loaded model:", type(model).__name__)
    print("MODEL_SOURCE:", MODEL_SOURCE)
    print("MODEL_LOAD_REF:", MODEL_LOAD_REF)




Model/tokenizer loading failed. Errors tried (most recent last):
 - fallback_name:microsoft/deberta-v3-base: OSError("We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.\nCheck your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.")
 - fallback_name:microsoft/deberta-v3-large: OSError("We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.\nCheck your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.")
 - fallback_name:microsoft/deberta-base: OSError("We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.\nCheck your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#off

In [5]:
def plot_confusion_matrix(preds, labels, title="Confusion Matrix"):
    preds = np.array(preds)
    labels = np.array(labels)

    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Greens", values_format="d")
    plt.title(title)
    plt.show()




In [6]:
def compute_metrics(preds, labels, device="cuda"):
    if len(preds.shape) > 1:
        preds = np.argmax(preds, axis=-1)
    qwk_score = cohen_kappa_score(labels, preds, weights="quadratic")
    acc_score = accuracy_score(labels, preds)
    return {"quadratic_weighted_kappa": qwk_score, "accuracy": acc_score}




In [7]:
def _load_train_prior(train_path: str, num_labels: int) -> np.ndarray:
    """
    Use train label distribution as deterministic prior for calibration in the fallback case.
    """
    df_tr = pd.read_csv(train_path, usecols=["score"])
    counts = (
        df_tr["score"]
        .value_counts()
        .reindex(range(1, num_labels + 1), fill_value=0)
        .values.astype(np.float64)
    )
    prior = counts / max(counts.sum(), 1.0)
    prior = np.clip(prior, 1e-12, 1.0)
    prior = prior / prior.sum()
    return prior


TRAIN_PRIOR = None
try:
    TRAIN_PRIOR = _load_train_prior(TRAIN_PATH, NUM_LABELS)
    print("Loaded train prior:", TRAIN_PRIOR.round(4), "sum=", float(TRAIN_PRIOR.sum()))
except Exception as e:
    print("Warning: could not load train prior:", repr(e))
    TRAIN_PRIOR = None




Loaded train prior: [0.0722 0.2728 0.3614 0.2287 0.0562 0.0087] sum= 1.0


In [8]:
def predict_essay_score(
    model, dataset, num_labels, batch_size, device, train_prior=None
):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model = model.to(device)
    model.eval()

    preds, essay_ids_all = [], []

    model_num_labels = getattr(getattr(model, "config", None), "num_labels", None)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting", unit="batch"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            essay_ids = batch["essay_id"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

            if model_num_labels == num_labels:
                pred_labels = torch.argmax(logits, axis=1).cpu().numpy() + 1
            else:
                if logits.shape[1] >= 2:
                    scalar = (logits[:, 1] - logits[:, 0]).float()
                else:
                    scalar = logits[:, 0].float()
                s = scalar.detach().cpu().numpy()
                preds.extend(s.tolist())
                essay_ids_all.extend(essay_ids)
                continue

            preds.extend(pred_labels.tolist())
            essay_ids_all.extend(essay_ids)

    preds = np.array(preds)
    essay_ids_all = np.array(essay_ids_all)

    chunk_results = pd.DataFrame({"essay_id": essay_ids_all, "raw_prediction": preds})

    aggregated_predictions = chunk_results.groupby("essay_id")["raw_prediction"].mean()
    agg_vals = aggregated_predictions.values

    if model_num_labels == num_labels:
        y_pred_aggregated = np.rint(agg_vals)
        final_scores = np.clip(y_pred_aggregated, 1, num_labels).astype(int)
    else:
        s = agg_vals.astype(np.float64)
        ranks = s.argsort().argsort().astype(np.float64)
        u = (ranks + 0.5) / max(len(s), 1.0)

        if train_prior is None:
            edges = np.linspace(0.0, 1.0, num_labels + 1)
        else:
            edges = np.concatenate([[0.0], np.cumsum(train_prior)])
            edges[-1] = 1.0

        # Change (score-improving correctness): np.searchsorted returns 0..num_labels indices here.
        # Map explicitly to 1..num_labels to avoid off-by-one shifts (which harms QWK).
        final_scores = np.searchsorted(edges, u, side="right")
        final_scores = np.clip(final_scores, 1, num_labels).astype(int)

    final_results_df = pd.DataFrame(
        {"essay_id": aggregated_predictions.index.values, "score": final_scores}
    )

    return final_results_df, preds




In [9]:
class TestEssayDataset(Dataset):
    def __init__(
        self,
        df,
        tokenizer,
        max_len: int = 512,
        overlap: int = 128,
        min_chunk_ratio: float = 0.3,
    ):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.overlap = overlap
        self.min_chunk_ratio = min_chunk_ratio

        self.num_special = tokenizer.num_special_tokens_to_add(pair=False)

        tag_tokens = tokenizer("[B]", add_special_tokens=False)["input_ids"]
        len_tag = len(tag_tokens)

        self.max_content_len = max_len - self.num_special - len_tag

        for _, row in df.iterrows():
            essay_id = row["essay_id"]
            base_tokens = tokenizer(row["full_text"], add_special_tokens=False)[
                "input_ids"
            ]

            step = self.max_content_len - overlap
            start = 0

            while start < len(base_tokens):
                end = start + self.max_content_len
                chunk = base_tokens[start:end]

                if (
                    len(chunk) < self.max_content_len * self.min_chunk_ratio
                    and start != 0
                ):
                    break

                chunk_with_tag = tag_tokens + chunk

                processed = tokenizer.build_inputs_with_special_tokens(chunk_with_tag)
                if (
                    isinstance(processed, (tuple, list))
                    and len(processed) == 2
                    and all(isinstance(x, list) for x in processed)
                ):
                    processed = processed[0]

                if len(processed) > self.max_len:
                    processed = processed[: self.max_len]

                pad_len = self.max_len - len(processed)
                if pad_len > 0:
                    processed += [tokenizer.pad_token_id] * pad_len

                self.samples.append((processed, essay_id))

                start += step
                if end >= len(base_tokens):
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_ids, essay_id = self.samples[idx]
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "essay_id": essay_id,
        }




In [10]:
class TrainEssayDataset(Dataset):
    def __init__(
        self,
        df,
        tokenizer,
        max_len: int = 512,
        overlap: int = 128,
        min_chunk_ratio: float = 0.3,
    ):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.overlap = overlap
        self.min_chunk_ratio = min_chunk_ratio

        self.num_special = tokenizer.num_special_tokens_to_add(pair=False)
        tag_tokens = tokenizer("[B]", add_special_tokens=False)["input_ids"]
        len_tag = len(tag_tokens)
        self.max_content_len = max_len - self.num_special - len_tag

        for _, row in df.iterrows():
            essay_id = row["essay_id"]
            label = int(row["score"])  # 1..6
            base_tokens = tokenizer(row["full_text"], add_special_tokens=False)[
                "input_ids"
            ]

            step = self.max_content_len - overlap
            start = 0
            while start < len(base_tokens):
                end = start + self.max_content_len
                chunk = base_tokens[start:end]

                if (
                    len(chunk) < self.max_content_len * self.min_chunk_ratio
                ) and start != 0:
                    break

                chunk_with_tag = tag_tokens + chunk

                processed = tokenizer.build_inputs_with_special_tokens(chunk_with_tag)
                if (
                    isinstance(processed, (tuple, list))
                    and len(processed) == 2
                    and all(isinstance(x, list) for x in processed)
                ):
                    processed = processed[0]

                if len(processed) > self.max_len:
                    processed = processed[: self.max_len]

                pad_len = self.max_len - len(processed)
                if pad_len > 0:
                    processed += [tokenizer.pad_token_id] * pad_len

                self.samples.append((processed, essay_id, label - 1))  # 0..5

                start += step
                if end >= len(base_tokens):
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_ids, essay_id, label = self.samples[idx]
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long),
            "essay_id": essay_id,
        }


def _freeze_backbone_except_classifier(model6: torch.nn.Module) -> int:
    trainable = 0
    for name, p in model6.named_parameters():
        p.requires_grad = False

        if any(
            k in name
            for k in [
                "classifier",
                "score",
                "classification",
                "pooler",
                "pre_classifier",
            ]
        ):
            p.requires_grad = True

        if (".encoder.layer." in name) and (
            name.endswith(".weight") or name.endswith(".bias")
        ):
            layer_idx = None
            try:
                parts = name.split(".encoder.layer.", 1)[1].split(".")
                layer_idx = int(parts[0])
            except Exception:
                layer_idx = None

            if layer_idx is not None and layer_idx >= 11:
                p.requires_grad = True

        if p.requires_grad:
            trainable += p.numel()
    return trainable


def _maybe_fit_head_on_train(
    model,
    tokenizer,
    train_path: str,
    device: str,
    force: bool = False,
    load_ref: str | None = None,
):
    model_num_labels = getattr(getattr(model, "config", None), "num_labels", None)
    if (not force) and (model_num_labels == NUM_LABELS):
        return model, False

    base_ref = load_ref or getattr(
        getattr(model, "config", None), "_name_or_path", None
    )
    if base_ref is None:
        return model, False

    try:
        cfg6 = AutoConfig.from_pretrained(
            base_ref, local_files_only=True, num_labels=NUM_LABELS
        )
        model6 = AutoModelForSequenceClassification.from_pretrained(
            base_ref, config=cfg6, local_files_only=True, ignore_mismatched_sizes=True
        )
    except Exception as e:
        print(
            "Could not construct 6-label model from base_ref; will keep calibration fallback. Error:",
            repr(e),
        )
        return model, False

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model6.resize_token_embeddings(len(tokenizer))

    torch.manual_seed(42)
    np.random.seed(42)

    df_tr = pd.read_csv(train_path, usecols=["essay_id", "full_text", "score"])

    max_train_essays = 6000
    if len(df_tr) > max_train_essays:
        df_tr = (
            df_tr.groupby("score", group_keys=False)
            .apply(
                lambda x: x.sample(
                    n=max(1, int(round(max_train_essays * len(x) / len(df_tr)))),
                    random_state=42,
                )
            )
            .reset_index(drop=True)
        )
        if len(df_tr) > max_train_essays:
            df_tr = df_tr.sample(n=max_train_essays, random_state=42).reset_index(
                drop=True
            )

    train_ds = TrainEssayDataset(df_tr, tokenizer, MAX_LEN, OVERLAP, MIN_CHUNK_RATIO)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    model6 = model6.to(device)
    model6.train()

    trainable = _freeze_backbone_except_classifier(model6)
    print(f"Trainable parameters after freezing backbone: {trainable}")

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model6.parameters()),
        lr=2e-4,
        weight_decay=0.01,
    )

    epochs = 1
    print(
        f"Fitting 6-label head (+last layer) on {len(df_tr)} essays ({len(train_ds)} chunks), epochs={epochs} ..."
    )
    for ep in range(epochs):
        pbar = tqdm(train_loader, desc=f"Train ep {ep+1}/{epochs}", unit="batch")
        for batch in pbar:
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            out = model6(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = out.loss
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=float(loss.detach().cpu().item()))
    model6.eval()
    return model6, True


if MODEL_AVAILABLE:
    model, DID_FIT = _maybe_fit_head_on_train(
        model,
        tokenizer,
        TRAIN_PATH,
        device,
        force=FORCE_HEAD_FIT,
        load_ref=MODEL_LOAD_REF,
    )
    if DID_FIT:
        MODEL_SOURCE = f"{MODEL_SOURCE} + fitted_6label_head"
        print("Updated MODEL_SOURCE:", MODEL_SOURCE)



In [11]:
df_test = pd.read_csv(TEST_PATH)

if MODEL_AVAILABLE:
    test_dataset = TestEssayDataset(
        df_test, tokenizer, MAX_LEN, OVERLAP, MIN_CHUNK_RATIO
    )
    print("Num test essays:", len(df_test), "Num chunks:", len(test_dataset))
else:
    test_dataset = None
    print(
        "MODEL_AVAILABLE=False -> will create fallback submission without running inference."
    )
    print("Num test essays:", len(df_test))



MODEL_AVAILABLE=False -> will create fallback submission without running inference.
Num test essays: 1731


In [12]:
sample_path = os.path.join(os.path.dirname(TEST_PATH), "sample_submission.csv")
sample_sub = (
    pd.read_csv(sample_path)
    if os.path.exists(sample_path)
    else pd.DataFrame({"essay_id": df_test["essay_id"].astype(str)})
)

if MODEL_AVAILABLE:
    test_results, _ = predict_essay_score(
        dataset=test_dataset,
        model=model,
        num_labels=NUM_LABELS,
        batch_size=BATCH_SIZE,
        device=device,
        train_prior=TRAIN_PRIOR,
    )

    print("Submission preview (model/calibrated):")
    print(test_results.head())
    print("Submission shape (model/calibrated):", test_results.shape)

    test_results = sample_sub[["essay_id"]].merge(
        test_results, on="essay_id", how="left"
    )
    test_results["score"] = test_results["score"].fillna(3).astype(int)
else:
    test_results = sample_sub.copy()
    test_results["score"] = 3

test_results = test_results[["essay_id", "score"]]
test_results["essay_id"] = test_results["essay_id"].astype(str)
test_results["score"] = test_results["score"].astype(int)

out_file = os.path.join(OUTPUT_PATH, "submission.csv")
test_results.to_csv(out_file, index=False)

print("Wrote:", out_file)
print(pd.read_csv(out_file).head())
print("Shape:", pd.read_csv(out_file).shape)
print("File exists:", os.path.exists(out_file), "Size:", os.path.getsize(out_file))
print("MODEL_SOURCE:", MODEL_SOURCE)

Wrote: /kaggle/working/submission.csv
  essay_id  score
0  d550b2d      3
1  0c10954      3
2  ef04816      3
3  88ab8d9      3
4  cee8c0f      3
Shape: (1731, 2)
File exists: True Size: 17325
MODEL_SOURCE: None
